[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SivanSchwartz/representation-learning-course/blob/master/tutoring/session01/solutions/03_lightning.ipynb)

# 03 · PyTorch Lightning: you write the methods, the Trainer calls them

Fell behind on 02? Run `git pull --no-edit` and open `tutoring/session01/solutions/02_bridge_to_pytorch.ipynb`.

In [1]:
# Setup: Colab install, data and loaders (the same as notebook 02). Just run it.
import io
import logging
import os
import subprocess
import sys
import urllib.request
import warnings

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning"], check=True)

import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Quieter output. num_workers=0 is on purpose: worker processes break notebooks on Windows and macOS.
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", ".*LeafSpec.*")
warnings.filterwarnings("ignore", ".*GPU available but not used.*")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

DATA_URL = "https://raw.githubusercontent.com/SivanSchwartz/representation-learning-course/master/tutoring/data/mnist_small.npz"


def load_mnist():
    """The local copy if there is one (your laptop), otherwise download it (Colab)."""
    for path in ["../data/mnist_small.npz", "../../data/mnist_small.npz"]:
        if os.path.exists(path):
            return dict(np.load(path))
    return dict(np.load(io.BytesIO(urllib.request.urlopen(DATA_URL).read())))


data = load_mnist()
X = torch.tensor(data["X_train"], dtype=torch.float32).reshape(-1, 784) / 255
y = torch.tensor(data["y_train"], dtype=torch.long)
X_test = torch.tensor(data["X_test"], dtype=torch.float32).reshape(-1, 784) / 255
y_test = torch.tensor(data["y_test"], dtype=torch.long)
torch.manual_seed(0)
train_loader = DataLoader(TensorDataset(X[:5000], y[:5000]), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X[5000:], y[5000:]), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

## 1. Build your own (tiny) Trainer

In notebook 02, the training loop called the model. Let's flip that around. A `MiniTrainer` owns the 🔧 plumbing, and it calls methods with **agreed names** on whatever model you give it:
- `model.configure_optimizers()`: "which optimizer do you want?"
- `model.training_step(batch, batch_idx)`: "here's a batch, give me the loss"

In [2]:
# ✍️ The plumbing from notebook 02, packed into a class. It never mentions Linear, ReLU or cross_entropy.
class MiniTrainer:
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs

    def fit(self, model, train_loader):
        optimizer = model.configure_optimizers()
        for epoch in range(self.max_epochs):
            for batch_idx, batch in enumerate(train_loader):
                loss = model.training_step(batch, batch_idx)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            print(f"epoch {epoch}: last loss {loss.item():.3f}")

In [3]:
# ✍️ The model holds only the 🔬 science, as methods with the names MiniTrainer expects
class MyNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

    def training_step(self, batch, batch_idx):
        x, y = batch
        return F.cross_entropy(self(x), y)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


MiniTrainer(max_epochs=2).fit(MyNN(), train_loader)

epoch 0: last loss 0.508
epoch 1: last loss 0.424


Notice what happened: **you never called `training_step`**. You wrote it, and the trainer called it. This is *inversion of control* ("don't call us, we'll call you"), the same idea as `forward` in `TinyModule`.

In [4]:
# 🤔 Predict: what happens if we give MiniTrainer a model that doesn't follow the agreement?
plain_model = nn.Sequential(nn.Linear(784, 10))
MiniTrainer(max_epochs=1).fit(plain_model, train_loader)

AttributeError: 'Sequential' object has no attribute 'configure_optimizers'

`'Sequential' object has no attribute 'configure_optimizers'`. The trainer relies on the method names. Lightning works the same way, so **a typo in a hook name** (`train_step`, `configure_optimizer`) means Lightning can't find your method. Depending on which hook it is, Lightning either raises an error like this one or quietly never calls your method.

## 2. Swap in the real thing: `LightningModule` + `pl.Trainer`

Change the parent class, add a `validation_step`, and give the model to Lightning's `Trainer`. Your methods stay the same.

In [5]:
# ✍️ Same methods. Only the parent class changes, and we add validation_step.
class LitNN(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

    def training_step(self, batch, batch_idx):
        x, y = batch
        return F.cross_entropy(self(x), y)

    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [6]:
# ✍️ logger=False for now: notebook 04 turns logging on.
torch.manual_seed(0)
lit_model = LitNN()
trainer = pl.Trainer(max_epochs=2, accelerator="cpu", logger=False, enable_checkpointing=False)
trainer.fit(lit_model, train_loader, val_loader)
trainer.callback_metrics

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00, 148.21it/s]

Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 239.66it/s]

Training: |          | 0/? [00:00<?, ?it/s]

... (336 outputs left out)

Epoch 1: 100%|██████████| 79/79 [00:00<00:00, 565.93it/s, val_loss=0.434]

Epoch 1: 100%|██████████| 79/79 [00:00<00:00, 564.77it/s, val_loss=0.434]

Epoch 1: 100%|██████████| 79/79 [00:00<00:00, 564.16it/s, val_loss=0.434]

{'val_loss': tensor(0.4337)}

Lightning also ran the validation loop for you, switched `eval()`/`train()` and `no_grad()` on and off at the right times, and printed a model summary and a progress bar. That's all the 🔧 plumbing from notebook 02, and the plumbing you didn't write in your exercise. `pl.LightningModule` is itself a child of `nn.Module`, so everything from notebook 02 still works: `lit_model(x)`, `lit_model.parameters()`, `print(lit_model)`.

## 3. Hooks: when does Lightning call what?

Besides the `*_step` methods, a `LightningModule` has dozens of empty hook methods (`on_fit_start`, `on_train_epoch_start`, …). Lightning calls them at fixed moments. Override one, and your code runs at that moment, just like overriding `forward` in notebook 01.

We inherit from `LitNN` and override five hooks to print when they're called:

In [7]:
# ✍️ A child class that only overrides hooks
class ChattyNN(LitNN):
    def on_fit_start(self):
        print("on_fit_start")

    def on_train_epoch_start(self):
        print(f"  on_train_epoch_start       epoch {self.current_epoch}")

    def on_validation_epoch_start(self):
        print(f"  on_validation_epoch_start  epoch {self.current_epoch}")

    def on_train_epoch_end(self):
        print(f"  on_train_epoch_end         epoch {self.current_epoch}")

    def on_fit_end(self):
        print("on_fit_end")

In [8]:
# 🤔 Predict: which hook prints right after on_fit_start? Training or validation?
quick = pl.Trainer(max_epochs=2, limit_train_batches=5, limit_val_batches=2, accelerator="cpu",
                   logger=False, enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False)
quick.fit(ChattyNN(), train_loader, val_loader)

on_fit_start


  on_validation_epoch_start  epoch 0


  on_train_epoch_start       epoch 0


  on_validation_epoch_start  epoch 0

  on_train_epoch_end         epoch 0
  on_train_epoch_start       epoch 1


  on_validation_epoch_start  epoch 1


  on_train_epoch_end         epoch 1
on_fit_end


Two surprises:
1. **Validation runs first.** It's the *sanity check*: before a long training run starts, Lightning runs two validation batches so a bug in `validation_step` crashes now, not after an hour of training. (`num_sanity_val_steps=0` turns it off.)
2. Each epoch's validation runs **before** `on_train_epoch_end`.

`self.current_epoch` is an attribute that the Trainer keeps up to date on *your* model. The course's `callbacks.py` uses the same hooks, only on a `Callback` class instead of the model.

### ✏️ Exercise 3: add a test step (≈6 min)

`TestableNN` inherits everything from `LitNN`. Add a `test_step(self, batch, batch_idx)` that:
1. computes the predictions, `self(x).argmax(dim=1)`;
2. computes the batch accuracy, `(preds == y).float().mean()`;
3. logs it with `self.log("test_acc", acc)`.

The check cell trains your model and calls `trainer.test(...)`, which calls your `test_step`.

In [9]:
class TestableNN(LitNN):
    def test_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x).argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("test_acc", acc)

In [10]:
# --- Check (do not edit) ---
torch.manual_seed(0)
testable = TestableNN()
check_trainer = pl.Trainer(max_epochs=2, accelerator="cpu", logger=False, enable_checkpointing=False,
                           enable_progress_bar=False, enable_model_summary=False)
check_trainer.fit(testable, train_loader, val_loader)
results = check_trainer.test(testable, test_loader, verbose=False)
assert "test_acc" in results[0], "Finish the exercise above first ☝️ (test_step should call self.log('test_acc', acc))"
assert results[0]["test_acc"] > 0.85, f"Test accuracy is only {results[0]['test_acc']:.3f}. Check how you compute preds and acc."
print(f"✅ Test accuracy: {results[0]['test_acc']:.3f}")

✅ Test accuracy: 0.879


## 4. Where's the training loop now?

Inside `pl.Trainer.fit`. It's the same loop as `MiniTrainer`'s, plus many more hooks, devices, precision, checkpoints, and early stopping. The arguments you pass to `pl.Trainer(...)` switch those features on and off, and **none of them require changing your model class**. For example, lec5's `train.py` builds its `Trainer` with:

```python
trainer = pl.Trainer(logger=logger, accelerator="auto", devices=1, max_epochs=num_epochs,
                     precision="16-mixed", callbacks=[EarlyStopping(monitor="val_loss"), ...])
```

## 5. The same idea for data: `LightningDataModule`

The model's methods describe *how to learn*. A `LightningDataModule`'s methods describe *where the data comes from*. The course's `MnistDataModule` (`lec2/datamodule.py` to `lec5/datamodule.py`) has this shape.

In [11]:
# ✍️ Data as hooks too: Lightning calls setup() once, then asks for each loader when it needs it
class SmallMnistDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=64):
        super().__init__()
        self.batch_size = batch_size

    def setup(self, stage):
        self.train_ds = TensorDataset(X[:5000], y[:5000])
        self.val_ds = TensorDataset(X[5000:], y[5000:])
        self.test_ds = TensorDataset(X_test, y_test)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=256)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=256)

In [12]:
# ✍️ Now fit() takes the data module instead of loaders, exactly like the course's train.py
dm = SmallMnistDataModule()
dm_trainer = pl.Trainer(max_epochs=1, accelerator="cpu", logger=False, enable_checkpointing=False, enable_model_summary=False)
dm_trainer.fit(LitNN(), datamodule=dm)

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00, 961.78it/s]

Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 916.19it/s]

Training: |          | 0/? [00:00<?, ?it/s]

... (166 outputs left out)

Validation DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 644.88it/s]

Epoch 0: 100%|██████████| 79/79 [00:00<00:00, 660.48it/s, val_loss=0.658]

Epoch 0: 100%|██████████| 79/79 [00:00<00:00, 659.50it/s, val_loss=0.658]

Epoch 0: 100%|██████████| 79/79 [00:00<00:00, 658.56it/s, val_loss=0.658]

🔄 **Checkpoint:** run `git pull --no-edit` to get `tutoring/session01/solutions/03_lightning.ipynb`.